# Lab 9.9 &mdash; Privacy: Minimize & Redact

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 5 &middot; Module 9 &mdash; Agents in Finance, Healthcare &amp; Cybersecurity**

### What you'll do
- Redact long identifiers (account/card numbers) from text
- Send the model only the fields the task needs
- Treat data handling as a first-class design constraint

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** these labs use the **real** LangChain (`langchain`, `langchain-core`, `langchain-ollama`). The **graded** cells assert only on the deterministic parts you build &mdash; grounding/citation/compute logic and, in the agent-assembly labs, tool wiring &amp; the read-only guardrail &mdash; and never call an LLM, so the lab always verifies offline. Cells marked **Optional &mdash; run it for real** call a live local model (`ollama run llama3.2:1b`, or Groq) and self-skip if none is reachable. You are building the **financial-report insight agent** &mdash; the client's Lab 5.1. It grounds &amp; cites every figure, gives **no advice**, and has **no trade tool** &mdash; a human analyst decides.

**Reference:** [Module 9 slides &mdash; Privacy, compliance & data handling](../../presentation/day5-module9-agents-in-industry.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 9 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket
WORK = "/tmp/biaa-lab-09-09"
os.makedirs(WORK, exist_ok=True)
def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening -- the optional live cells self-skip when it isn't."""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False
print("Working dir:", WORK, "| Ollama reachable:", ollama_up())

## Concept
High-stakes agents run on the most sensitive data there is (deck slide 16). Two disciplines: **minimize**
&mdash; send the model only the fields the task needs, not the whole record &mdash; and **redact** &mdash;
mask identifiers (account numbers, card numbers) before they leave your systems. Less exposed data is
less risk. It's why Day 1 started on **local models**.

In [ ]:
# We mask any run of 6+ consecutive digits (account/card numbers), keeping short numbers like a year.
print("redact 'acct 1234567890 for FY2026' -> mask the long number, keep 2026")

## Your Turn
Complete `redact_account` (mask long digit runs) and `minimize` (keep only needed fields).

In [ ]:
def redact_account(text):
    out, run = [], ""
    for ch in text + " ":            # trailing space flushes the final run
        if ch.isdigit():
            run += ch
        else:
            if ___:            # TODO: run is a long number (6+ digits)
                out.append("[REDACTED]")
            else:
                out.append(run)
            run = ""
            out.append(ch)
    return "".join(out).rstrip()

def minimize(record, needed_fields):
    # send the model ONLY the fields the task needs -- not the whole record
    return ___   # TODO: build a dict of just the needed_fields that exist in record

try:
    print(redact_account('acct 1234567890 for FY2026'))
    rec = {'name': 'ACME', 'revenue': 120.0, 'account': '1234567890', 'ssn': '999-99-9999'}
    print(minimize(rec, ['name', 'revenue']))
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("a 10-digit account number is masked", lambda: "[REDACTED]" in redact_account("acct 1234567890 here"))
expect_true("a short number (a year) is kept", lambda: "2026" in redact_account("for FY2026"))
expect_true("the redacted text has no long digit run", lambda: not any(len(w) >= 6 and w.isdigit() for w in redact_account("acct 1234567890").split()))
expect_true("minimize keeps only the needed fields", lambda: minimize({"name": "ACME", "revenue": 120.0, "account": "1234567890"}, ["name", "revenue"]) == {"name": "ACME", "revenue": 120.0})
expect_true("minimize drops the sensitive fields", lambda: "account" not in minimize({"name": "ACME", "account": "1234567890", "ssn": "999-99-9999"}, ["name"]))

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

---
### Key takeaway
Minimize what you send and redact identifiers before they leave your systems. Data handling is a first-class design constraint -- decide where the data can go before you build, which is why this course runs on local models.

[&#8592; All Module 9 labs](./index.html) &nbsp;&middot;&nbsp; [Module 9 slides](../../presentation/day5-module9-agents-in-industry.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>